# Create the Procurement 360 Delta table

Use this notebook in **Oracle AI Data Platform Workbench** to read the procurement CSV from a managed volume and overwrite a managed Delta table for the agent's Spark SQL tool.

**Result:** `default.default.procurement_information`

## Before you run

- Attach a running Spark cluster to this notebook.
- In **Master Catalog → default → default → Volumes → pwc**, confirm that `Generic_Airline_Procurement_360.csv` is present.
- Volume paths are case-sensitive. If your volume or file name differs, change `csv_path` in Step 1.
- The commented XLSX path is only a reference; this notebook reads the CSV.

Run every cell from top to bottom. The overwrite operation is intentional for this synthetic lab dataset.

## Step 1 — Configure Spark, the volume path and the target table

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

csv_path = "/Volumes/default/default/pwc/Generic_Airline_Procurement_360.csv"
volume_root = str(Path(csv_path).parent)
delta_table = "default.default.procurement_information"

# Reference only; this notebook uses the CSV above.
# xlsx_path = "/Volumes/default/default/pwc/Generic_Airline_Procurement_SQL_Demo_Data.xlsx"

print(f"Source CSV : {csv_path}")
print(f"Delta table: {delta_table}")

## Step 2 — Check the volume path before reading

This check produces a clear error when the volume name, folder or CSV filename is wrong. If it fails, copy the exact path from the Workbench volume and update Step 1.

In [ ]:
volume_directory = Path(volume_root)
csv_file = Path(csv_path)

if not volume_directory.is_dir():
    raise FileNotFoundError(
        f"Volume path was not found: {volume_root}. "
        "Check the catalog, schema and volume names in Master Catalog."
    )

available_files = sorted(item.name for item in volume_directory.iterdir())
print(f"Files visible in {volume_root}:")
for filename in available_files:
    print(f"  - {filename}")

if not csv_file.is_file():
    raise FileNotFoundError(
        f"Expected CSV was not found: {csv_path}. "
        "Check spelling and capitalization against the file list above."
    )

print()
print(f"Path check passed: {csv_path}")

## Step 3 — Read the CSV with Spark

In [ ]:
procurement_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

source_row_count = procurement_df.count()
print(f"Rows read   : {source_row_count:,}")
print(f"Columns read: {len(procurement_df.columns)}")
procurement_df.printSchema()
procurement_df.show(5, truncate=False)

## Step 4 — Validate the columns required by the agent

In [ ]:
required_columns = {
    "snapshot_time",
    "req_id",
    "department",
    "urgency",
    "estimated_value_usd",
    "approval_status",
    "supplier_risk_tier",
    "contract_status",
    "receipt_quality_status",
    "invoice_match_status",
    "attention_reasons",
}

missing_columns = sorted(required_columns.difference(procurement_df.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing from the CSV: {missing_columns}")

if source_row_count == 0:
    raise ValueError("The CSV was read successfully but contains no data rows.")

print("Schema validation passed.")

## Step 5 — Write the managed Delta table

`overwrite` and `overwriteSchema` make the lab repeatable. Do not use this pattern for a production table until its retention and change-control requirements are defined.

In [ ]:
(
    procurement_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(delta_table)
)

print(f"Delta table written successfully: {delta_table}")

## Step 6 — Validate the Delta table

In [ ]:
target_row_count = spark.table(delta_table).count()
print(f"Source rows: {source_row_count:,}")
print(f"Target rows: {target_row_count:,}")

if target_row_count != source_row_count:
    raise ValueError(
        f"Row-count mismatch: source={source_row_count}, target={target_row_count}"
    )

spark.sql(f"SELECT COUNT(*) AS total_records FROM {delta_table}").show()
spark.sql(f"SELECT * FROM {delta_table} LIMIT 10").show(truncate=False)
print("Delta table validation passed.")

## Step 7 — Validate the query used by the agent SQL tool

The agent's SQL tool should use the **Spark SQL** dialect, the running Spark cluster and this managed standard-catalog Delta table. The runtime parameter is `req_id`.

In [ ]:
test_req_id = "REQ-0001"

spark.sql(
    f"""
    SELECT req_id,
           department,
           urgency,
           estimated_value_usd,
           approval_status,
           supplier_risk_tier,
           contract_status,
           receipt_quality_status,
           invoice_match_status,
           attention_reasons,
           snapshot_time
    FROM {delta_table}
    WHERE LOWER(req_id) = LOWER('{test_req_id}')
    """
).show(truncate=False)

## SQL Tool configuration

In the agent canvas:

1. Drag the **SQL** tool to the Executor Agent.
2. Select **Spark SQL** as the query dialect.
3. Select the running Spark cluster.
4. Browse to `default → default → procurement_information`.
5. Paste the query below and define `req_id` as a String parameter.
6. Set **Max rows to return** to `10`, then test with `REQ-0001`.

In [ ]:
agent_sql_query = """
SELECT req_id,
       department,
       urgency,
       estimated_value_usd,
       approval_status,
       supplier_risk_tier,
       contract_status,
       receipt_quality_status,
       invoice_match_status,
       attention_reasons,
       snapshot_time
FROM default.default.procurement_information
WHERE LOWER(req_id) = LOWER('{{req_id}}')
""".strip()

print(agent_sql_query)

## Completion checklist

- The path check finds `Generic_Airline_Procurement_360.csv`.
- Spark reads a non-zero row count and all required columns.
- `default.default.procurement_information` is visible in **Master Catalog → Tables** with format **Delta**.
- Source and target row counts match.
- The `REQ-0001` validation query returns a row.
- The SQL tool test succeeds before the full agent is tested in Playground.

**Optional exercise:** change `test_req_id` to another requisition from the preview and rerun Step 7.